### Введите наименование акутального файла all velo формата geojson

In [ ]:
file_path = "All_velo_02_10_2026.geojson"
gdf = gpd.read_file(file_path)


#### Ниже будет произведен расчет параметров велоинфраструктуры по округам

### Расчет велоинфраструктуры в парках

In [4]:
import geopandas as gpd
import pandas as pd


okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


filtered = gdf[
    (gdf["OKRUG"].isin(okrugs)) &
    (gdf["STATUS"].str.lower() == "парк") &
    (gdf["Duble"] != 2)
]

grouped = (
    filtered.groupby("OKRUG")["LEN"]
    .sum()
    .reindex(okrugs)
    .fillna(0) / 1000
)

grouped["Общий итог"] = grouped.sum()

print(grouped.round(2))

OKRUG
ВАО            63.55
ЗАО            49.75
ЗелАО           0.43
САО            13.51
СВАО           27.94
СЗАО           21.24
ТиНАО          12.46
ЦАО             9.63
ЮАО            42.14
ЮВАО           32.96
ЮЗАО           26.65
Общий итог    300.25
Name: LEN, dtype: float64


### Расчет общей постоянной велоинфраструктуры по округам

In [5]:
import geopandas as gpd
import pandas as pd

okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


# Очистка
gdf["STATUS"] = gdf["STATUS"].astype(str).str.strip().str.lower()
gdf["ZONE"] = gdf["ZONE"].astype(str).str.strip().str.lower()
gdf["OKRUG"] = gdf["OKRUG"].astype(str).str.strip()
gdf["Duble"] = pd.to_numeric(gdf["Duble"], errors="coerce")

# Фильтрация: только постоянные, нужные зоны, не дубли, и валидные округа
filtered = gdf[
    (gdf["STATUS"] == "постоянная") &
    (gdf["ZONE"].isin(["проезжая часть", "тротуар"])) &
    (gdf["Duble"] != 2) &
    (gdf["OKRUG"].isin(okrugs))
]

# Группировка и перевод в км
pivot = (
    filtered
    .groupby(["OKRUG", "ZONE"])["LEN"]
    .sum()
    .unstack()
    .reindex(index=okrugs, columns=["проезжая часть", "тротуар"])
    .fillna(0)
    / 1000
)

# Переименование колонок
pivot = pivot.rename(columns={
    "проезжая часть": "Проезжая часть",
    "тротуар": "Тротуар"
})

# Общий итог
pivot.loc["Общий итог"] = pivot.sum()

# Вывод
print(pivot.round(2))


ZONE        Проезжая часть  Тротуар
OKRUG                              
ВАО                  15.57    12.17
ЗАО                  22.55    57.61
ЗелАО                 0.00     2.64
САО                  10.03    20.75
СВАО                  8.31     7.03
СЗАО                  4.52     7.77
ТиНАО                 0.00    51.52
ЦАО                  28.95    13.64
ЮАО                   7.79    25.54
ЮВАО                 12.36    22.42
ЮЗАО                  1.98     8.05
Общий итог          112.05   229.13


### Расчет новой постоянной велоинфраструктуры, реализованной в 2026 году

In [6]:
import geopandas as gpd
import pandas as pd

okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


# Очистка
gdf["STATUS"] = gdf["STATUS"].astype(str).str.strip().str.lower()
gdf["ZONE"] = gdf["ZONE"].astype(str).str.strip().str.lower()
gdf["OKRUG"] = gdf["OKRUG"].astype(str).str.strip()
gdf["YEAR"] = pd.to_numeric(gdf["YEAR"], errors="coerce")
gdf["Duble"] = pd.to_numeric(gdf["Duble"], errors="coerce")

# Фильтрация
filtered = gdf[
    (gdf["YEAR"] == 2026) &
    (gdf["STATUS"] == "постоянная") &
    (gdf["ZONE"].isin(["проезжая часть", "тротуар"])) &
    (gdf["Duble"] != 2) &
    (gdf["OKRUG"].isin(okrugs))
]

# Группировка и перевод в километры
pivot = (
    filtered
    .groupby(["OKRUG", "ZONE"])["LEN"]
    .sum()
    .unstack()
    .reindex(index=okrugs, columns=["проезжая часть", "тротуар"])
    .fillna(0)
    / 1000
)

# Переименование столбцов
pivot = pivot.rename(columns={
    "проезжая часть": "Проезжая часть",
    "тротуар": "Тротуар"
})

# Добавляем строку "Общий итог"
pivot.loc["Общий итог"] = pivot.sum()

# Вывод результата
print(f'В 2026 году реализовано: \n\n {pivot.round(2)}')

В 2026 году реализовано: 

 ZONE        Проезжая часть  Тротуар
OKRUG                              
ВАО                   0.00     2.36
ЗАО                   7.12     4.37
ЗелАО                 0.00     0.00
САО                   6.48     2.15
СВАО                  0.00     2.66
СЗАО                  0.84     0.17
ТиНАО                 0.00     0.00
ЦАО                   6.78     1.55
ЮАО                   2.36     0.00
ЮВАО                  2.49     0.34
ЮЗАО                  0.00     0.00
Общий итог           26.07    13.62


### Расчет общей временной велоинфраструктуры

In [7]:
import geopandas as gpd
import pandas as pd

okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


# Очистка строк
gdf["OKRUG"] = gdf["OKRUG"].astype(str).str.strip()
gdf["TYPE"] = gdf["TYPE"].astype(str).str.strip().str.lower()
gdf["STATUS"] = gdf["STATUS"].astype(str).str.strip().str.lower()

# Фильтрация
filtered = gdf[
    (gdf["OKRUG"].isin(okrugs)) &
    (gdf["STATUS"].isin(["временная"])) &
    (gdf["TYPE"].isin([
        "оранжевая временная велополоса",
        "белая временная велополоса",
        "совмещенное движение",
        "велозона",
        "совмещенная велопешеходная дорожка"
    ])) &
    (gdf["Duble"] != 2)
].copy()

# Приводим метры в километры
filtered["LEN_KM"] = filtered["LEN"] / 1000

# Сводная таблица: по округам и типам
pivot = (
    filtered
    .groupby(["OKRUG", "TYPE"])["LEN_KM"]
    .sum()
    .unstack(fill_value=0)
    .reindex(okrugs)
)

# Переименование столбцов
pivot = pivot.rename(columns={
    "оранжевая временная велополоса": "Оранжевая временная",
    "белая временная велополоса": "Белая временная",
    "совмещенное движение": "Совмещенное движение (временная)",
    "велозона": "Велозона (временная)",
    "совмещенная велопешеходная дорожка": "Совмещенная велопешеходная дорожка (временная)"
})

# Итоговая строка
pivot.loc["Общий итог"] = pivot.sum()

# Вывод
print(pivot.round(2))

TYPE        Белая временная  Велозона (временная)  \
OKRUG                                               
ВАО                    2.80                  0.00   
ЗАО                    1.56                  0.00   
ЗелАО                   NaN                   NaN   
САО                    8.92                  0.00   
СВАО                  10.25                  0.00   
СЗАО                   8.26                  0.00   
ТиНАО                   NaN                   NaN   
ЦАО                   18.38                  0.00   
ЮАО                   26.96                  0.00   
ЮВАО                  14.66                  0.00   
ЮЗАО                  13.20                  0.42   
Общий итог           104.99                  0.42   

TYPE        Совмещенное движение (временная)  
OKRUG                                         
ВАО                                     0.00  
ЗАО                                     0.00  
ЗелАО                                    NaN  
САО                   

### Расчет новой временной велоинфраструктуры, реализованной в 2026 году

In [ ]:
import geopandas as gpd
import pandas as pd


# Список округов в нужном порядке
okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


# Очистка строк и приведение к нужному формату
gdf["TYPE"] = gdf["TYPE"].astype(str).str.strip().str.lower()
gdf["STATUS"] = gdf["STATUS"].astype(str).str.strip().str.lower()
gdf["OKRUG"] = gdf["OKRUG"].astype(str).str.strip()
gdf["YEAR"] = pd.to_numeric(gdf["YEAR"], errors="coerce")
gdf["Duble"] = pd.to_numeric(gdf["Duble"], errors="coerce")

# Типы временной велоинфраструктуры
temporary_types = [
    "оранжевая временная велополоса",
    "белая временная велополоса",
    "совмещенное движение",
    "велозона",
    "совмещенная велопешеходная дорожка"
]

# Фильтрация
filtered = gdf[
    (gdf["YEAR"] == 2026) &
    (gdf["STATUS"] == "временная") &
    (gdf["TYPE"].isin(temporary_types)) &
    (gdf["Duble"] != 2) &
    (gdf["OKRUG"].isin(okrugs))
].copy()

# Перевод в километры
filtered["LEN_KM"] = filtered["LEN"] / 1000

# Сводная таблица: округ × тип
pivot = (
    filtered
    .groupby(["OKRUG", "TYPE"])["LEN_KM"]
    .sum()
    .unstack(fill_value=0)
    .reindex(okrugs)
)

# Переименование столбцов
pivot = pivot.rename(columns={
    "оранжевая временная велополоса": "Оранжевая временная",
    "белая временная велополоса": "Белая временная",
    "совмещенное движение": "Совмещенное движение (временная)",
    "велозона": "Велозона (временная)",
    "совмещенная велопешеходная дорожка": "Совмещенная велопешеходная дорожка (временная)"
})

# Итог
pivot.loc["Общий итог"] = pivot.sum()

# Вывод
print(f'В 2026 году реализовано: \n\n {pivot.round(2)}')


В 2026 году реализовано: 

 TYPE        Белая временная
OKRUG                      
ВАО                     NaN
ЗАО                     NaN
ЗелАО                   NaN
САО                     NaN
СВАО                    NaN
СЗАО                    NaN
ТиНАО                   NaN
ЦАО                    0.68
ЮАО                     NaN
ЮВАО                    NaN
ЮЗАО                    NaN
Общий итог             0.68


### Расчет велоинфраструктуры по улично-дорожной сети (УДС)

In [9]:
import geopandas as gpd
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


gdf["STATUS"] = gdf["STATUS"].astype(str).str.strip().str.lower()
gdf["ZONE"] = gdf["ZONE"].astype(str).str.strip().str.lower()
gdf["OKRUG"] = gdf["OKRUG"].astype(str).str.strip()
gdf["Duble"] = pd.to_numeric(gdf["Duble"], errors="coerce")

filtered = gdf[
    (gdf["STATUS"] != "парк") &
    (gdf["ZONE"].isin(["проезжая часть", "тротуар"])) &
    (gdf["Duble"] != 2) &
    (gdf["OKRUG"].isin(okrugs))
].copy()

filtered["LEN_KM"] = filtered["LEN"] / 1000

pivot_raw = (
    filtered
    .groupby(["OKRUG", "ZONE"])["LEN_KM"]
    .sum()
    .unstack()
    .reindex(index=okrugs, columns=["проезжая часть", "тротуар"])
    .fillna(0)
)

def round_half_up(val):
    return float(Decimal(val).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

pivot = pivot_raw.applymap(round_half_up)

pivot.loc["Общий итог"] = pivot.sum()

pivot = pivot.rename(columns={
    "проезжая часть": "Проезжая часть",
    "тротуар": "Тротуар"
})

print(pivot)


ZONE        Проезжая часть  Тротуар
OKRUG                              
ВАО                  18.37    12.17
ЗАО                  24.11    57.61
ЗелАО                 0.00     2.64
САО                  18.95    20.75
СВАО                 18.55     7.03
СЗАО                 12.78     7.77
ТиНАО                 0.00    51.52
ЦАО                  49.28    13.64
ЮАО                  35.35    25.54
ЮВАО                 27.01    22.42
ЮЗАО                 15.60     8.05
Общий итог          220.00   229.14


C:\Users\User\AppData\Local\Temp\ipykernel_15256\489230530.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot = pivot_raw.applymap(round_half_up)


### Итоговая протяженность велдоинфраструктуры по тротуарам и проезжей части, включая парки

In [ ]:
import geopandas as gpd
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


gdf["ZONE"] = gdf["ZONE"].astype(str).str.strip().str.lower()
gdf["OKRUG"] = gdf["OKRUG"].astype(str).str.strip()
gdf["Duble"] = pd.to_numeric(gdf["Duble"], errors="coerce")

# Фильтрация
filtered = gdf[
    (gdf["ZONE"].isin(["проезжая часть", "тротуар"])) &
    (gdf["Duble"] != 2) &
    (gdf["OKRUG"].isin(okrugs))
].copy()

# Перевод длины в км
filtered["LEN_KM"] = filtered["LEN"] / 1000

# Группировка
pivot_raw = (
    filtered
    .groupby(["OKRUG", "ZONE"])["LEN_KM"]
    .sum()
    .unstack()
    .reindex(index=okrugs, columns=["проезжая часть", "тротуар"])
    .fillna(0)
)

# Округление математическое
def round_half_up(val):
    return float(Decimal(val).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

pivot = pivot_raw.applymap(round_half_up)

# Общий итог
pivot.loc["Общий итог"] = pivot.sum()

# Переименование
pivot.columns = ["Проезжая часть", "Тротуар"]

# Вывод
print(pivot)

            Проезжая часть  Тротуар
OKRUG                              
ВАО                  18.37    75.72
ЗАО                  24.11   107.36
ЗелАО                 0.43     2.64
САО                  18.95    34.26
СВАО                 18.55    34.97
СЗАО                 12.78    29.01
ТиНАО                 0.00    63.97
ЦАО                  49.28    23.27
ЮАО                  35.35    67.68
ЮВАО                 27.01    55.38
ЮЗАО                 15.60    34.69
Общий итог          220.43   528.95


C:\Users\User\AppData\Local\Temp\ipykernel_22264\2603856988.py:41: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivot = pivot_raw.applymap(round_half_up)


### Перепроверим общий итог (сверить с итоговой протяженностью в таблице "СВОД вело")

In [ ]:
import geopandas as gpd
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

okrugs = [
    "ВАО", "ЗАО", "ЗелАО", "САО", "СВАО",
    "СЗАО", "ТиНАО", "ЦАО", "ЮАО", "ЮВАО", "ЮЗАО"
]


# Фильтрация: нужные округа, Duble ≠ 2
filtered = gdf[(gdf["OKRUG"].isin(okrugs)) & (gdf["Duble"] != 2)]

# Сумма по округам (в км)
grouped = (
    filtered.groupby("OKRUG")["LEN"]
    .sum()
    .reindex(okrugs)
    .fillna(0) / 1000
)

# Округление по математике
def round_half_up(val):
    return float(Decimal(val).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP))

grouped_rounded = grouped.map(round_half_up)

# Подсчёт и добавление общего итога
total = round_half_up(grouped_rounded.sum())
grouped_rounded.loc["Общий итог"] = total

# Вывод
print(grouped_rounded)

OKRUG
ВАО            94.09
ЗАО           131.47
ЗелАО           3.07
САО            53.20
СВАО           53.52
СЗАО           41.80
ТиНАО          63.97
ЦАО            72.55
ЮАО           103.03
ЮВАО           82.39
ЮЗАО           50.30
Общий итог    749.39
Name: LEN, dtype: float64


### Выгрузка районов Москвы в excel-файле с протяженностью велоинфраструктуры

In [ ]:
import geopandas as gpd  
import pandas as pd
from decimal import Decimal, ROUND_HALF_UP

gdf["DISTRICT_2"] = gdf["DISTRICT_2"].astype(str).str.strip()
gdf["Duble"] = pd.to_numeric(gdf["Duble"], errors="coerce")

# Фильтрация
filtered = gdf[gdf["Duble"] != 2].copy()

# Перевод в километры
filtered["LEN_KM"] = filtered["LEN"] / 1000

# Группировка по районам
district_sum_raw = (
    filtered.groupby("DISTRICT_2")["LEN_KM"]
    .sum()
    .sort_values(ascending=False)
)

# Округление и форматирование (запятая вместо точки)
def format_with_comma(val):
    rounded = Decimal(val).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    return str(rounded).replace(".", ",")

district_sum = district_sum_raw.map(format_with_comma)

# Добавление строки "Общий итог"
district_sum.loc["Общий итог"] = format_with_comma(district_sum_raw.sum())

# Преобразуем в DataFrame
district_df = district_sum.reset_index()
district_df.columns = ["Район", "Длина (км)"]

# Сохраняем в Excel
output_file = "Длины по районам.xlsx"
district_df.to_excel(output_file, index=False)

print(f"✅ Файл успешно сохранён как: {output_file}")


✅ Файл успешно сохранён как: Длины по районам.xlsx
